# Function 8: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [ ]:
import numpy as np

input_data = np.load('initial_data/function_8/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.107284, 0.230012, 0.005809, 0.110406, 0.944752, 0.761346, 0.292029, 0.513328],   # week 1
    [0.001554, 0.874702, 0.057194, 0.944468, 0.772538, 0.226518, 0.257570, 0.752127]    # week 2
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


In [ ]:
output_data = np.load('initial_data/function_8/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    9.8491858885566,    # week 1
    8.7242298801731     # week 2
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]])
actual_output = 8.798300000000001

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 8
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


In [ ]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


## Gaussian process surrogate + expected improvement for minimisation

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

# Standardise inputs and outputs for numerical stability.
X = input_data.copy()
y = output_data.copy()

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2), nu=2.5) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1))

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=False, n_restarts_optimizer=20, random_state=0)
gp.fit(X_scaled, y_scaled)

print("Fitted kernel:", gp.kernel_)
print("Best observed y:", y.min())
print("Best observed x:", X[np.argmin(y)])


In [ ]:
def expected_improvement_min(X_candidates, gp, y_best_scaled, xi=0.01):
    """Expected improvement for minimisation in scaled y-space."""
    mu, sigma = gp.predict(X_candidates, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    improvement = y_best_scaled - mu - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

# Random candidate search in [0, 1]^d. Increase n_candidates for a more exhaustive search.
rng = np.random.default_rng(0)
n_candidates = 20000 if d <= 4 else 50000
candidates = rng.random((n_candidates, d))
candidates_scaled = x_scaler.transform(candidates)

y_best_scaled = y_scaled.min()
ei = expected_improvement_min(candidates_scaled, gp, y_best_scaled, xi=0.01)
mu_scaled, std_scaled = gp.predict(candidates_scaled, return_std=True)
mu = y_scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
std = std_scaled * y_scaler.scale_[0]

results = pd.DataFrame(candidates, columns=[f"x{i+1}" for i in range(d)])
results["pred_mean"] = mu
results["pred_std"] = std
results["expected_improvement"] = ei
results = results.sort_values("expected_improvement", ascending=False)

display(results.head(10))

best_next = results.iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(dtype=float)
print("Suggested next query:", np.round(best_next, 6))
print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(best_next)))


In [ ]:
# Optional 2D visualisation only when d=2
if d == 2:
    grid_res = 150
    xx, yy = np.meshgrid(np.linspace(0, 1, grid_res), np.linspace(0, 1, grid_res))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = x_scaler.transform(grid)
    mu_grid_scaled, std_grid_scaled = gp.predict(grid_scaled, return_std=True)
    mu_grid = y_scaler.inverse_transform(mu_grid_scaled.reshape(-1, 1)).reshape(grid_res, grid_res)
    ei_grid = expected_improvement_min(grid_scaled, gp, y_best_scaled).reshape(grid_res, grid_res)

    plt.figure(figsize=(7, 6))
    cf = plt.contourf(xx, yy, mu_grid, levels=40)
    plt.colorbar(cf, label="GP predicted mean")
    plt.scatter(input_data[:, 0], input_data[:, 1], c=output_data, edgecolors="black", s=80)
    plt.scatter(best_next[0], best_next[1], marker="*", s=250, edgecolors="black", label="suggested next")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title("GP predicted mean")
    plt.legend(); plt.show()

    plt.figure(figsize=(7, 6))
    cf = plt.contourf(xx, yy, ei_grid, levels=40)
    plt.colorbar(cf, label="Expected improvement")
    plt.scatter(input_data[:, 0], input_data[:, 1], c="white", edgecolors="black", s=80)
    plt.scatter(best_next[0], best_next[1], marker="*", s=250, edgecolors="black", label="suggested next")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title("Expected improvement for minimisation")
    plt.legend(); plt.show()
